In [ ]:
# ============================================================
# FULL GRAD-CAM VISUALIZATION CODE
# Fusion Model: DenseNet121 + EfficientNetB0 + SE + CBAM
# ============================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from PIL import Image
from glob import glob
from torchvision import models, transforms

# ============================================================
# PATHS
# ============================================================

# trained model path
model_path = "/kaggle/input/datasets/sabbir4724/modelpath/best_model.pth"

# dataset folder
dataset_path = "/kaggle/input/datasets/sabbir4724/sabbirs-dataset"

# output folder
output_dir = "/kaggle/working/output"

os.makedirs(output_dir, exist_ok=True)

# ============================================================
# RANDOM IMAGE SELECTION
# ============================================================

image_extensions = [
    "*.jpg",
    "*.jpeg",
    "*.png"
]

all_images = []

for ext in image_extensions:

    all_images.extend(
        glob(
            os.path.join(
                dataset_path,
                "**",
                ext
            ),
            recursive=True
        )
    )

# random image
image_path = random.choice(all_images)

print("Selected Image:")
print(image_path)

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# ============================================================
# CLASS NAMES
# ============================================================

class_names = [
    "Normal",
    "Pneumonia Bacterial",
    "Pneumonia Viral"
]

# ============================================================
# SE BLOCK
# ============================================================

class SEBlock(nn.Module):

    def __init__(self, channels, reduction=16):

        super().__init__()

        self.fc1 = nn.Linear(
            channels,
            channels // reduction
        )

        self.fc2 = nn.Linear(
            channels // reduction,
            channels
        )

        self.relu = nn.ReLU()

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        b, c, _, _ = x.size()

        y = x.mean(dim=(2,3))

        y = self.fc1(y)

        y = self.relu(y)

        y = self.fc2(y)

        y = self.sigmoid(y).view(
            b,
            c,
            1,
            1
        )

        return x * y.expand_as(x)

# ============================================================
# CBAM
# ============================================================

class CBAM(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16,
        kernel_size=7
    ):

        super().__init__()

        self.mlp = nn.Sequential(

            nn.Linear(
                channels,
                channels // reduction
            ),

            nn.ReLU(),

            nn.Linear(
                channels // reduction,
                channels
            )
        )

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size,
            padding=kernel_size // 2
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        # Channel attention
        avg_pool = x.mean(dim=(2,3))

        max_pool = torch.amax(
            x,
            dim=(2,3)
        )

        ca = self.mlp(avg_pool) + self.mlp(max_pool)

        ca = torch.sigmoid(ca).view(
            x.size(0),
            x.size(1),
            1,
            1
        )

        x = x * ca

        # Spatial attention
        avg_out = x.mean(
            dim=1,
            keepdim=True
        )

        max_out = torch.amax(
            x,
            dim=1,
            keepdim=True
        )

        sa = torch.cat(
            [avg_out, max_out],
            dim=1
        )

        sa = self.sigmoid(
            self.conv(sa)
        )

        x = x * sa

        return x

# ============================================================
# FUSION MODEL
# ============================================================

class FusionModel(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        # DenseNet121
        self.densenet = models.densenet121(
            pretrained=False
        )

        self.densenet.classifier = nn.Identity()

        self.se_d = SEBlock(1024)

        self.head_d = nn.Sequential(

            nn.Linear(1024,512),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.ReLU(),

            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        # EfficientNetB0
        self.efficientnet = models.efficientnet_b0(
            pretrained=False
        )

        self.efficientnet.classifier = nn.Identity()

        self.se_e = SEBlock(1280)

        self.head_e = nn.Sequential(

            nn.Linear(1280,512),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.ReLU(),

            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        # Fusion
        fusion_dim = 512

        self.cbam = CBAM(
            fusion_dim
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                fusion_dim,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(self, x):

        # DenseNet branch
        f_d = self.densenet(x)

        f_d = f_d.unsqueeze(-1).unsqueeze(-1)

        f_d = self.se_d(f_d)

        f_d = f_d.squeeze(-1).squeeze(-1)

        f_d = self.head_d(f_d)

        # EfficientNet branch
        f_e = self.efficientnet(x)

        f_e = f_e.unsqueeze(-1).unsqueeze(-1)

        f_e = self.se_e(f_e)

        f_e = f_e.squeeze(-1).squeeze(-1)

        f_e = self.head_e(f_e)

        # Fusion
        f = torch.cat(
            [f_d, f_e],
            dim=1
        )

        f = f.unsqueeze(-1).unsqueeze(-1)

        f = self.cbam(f)

        f = f.squeeze(-1).squeeze(-1)

        out = self.classifier(f)

        return out

# ============================================================
# LOAD MODEL
# ============================================================

model = FusionModel(
    num_classes=3
).to(device)

model.load_state_dict(
    torch.load(
        model_path,
        map_location=device
    )
)

model.eval()

print("Model loaded successfully")

# ============================================================
# IMAGE TRANSFORM
# ============================================================

transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

# ============================================================
# LOAD IMAGE
# ============================================================

image = Image.open(
    image_path
).convert("RGB")

original_image = image.resize((224,224))

original_np = np.array(
    original_image
)

input_tensor = transform(
    original_image
).unsqueeze(0).to(device)

# ============================================================
# HOOKS
# ============================================================

gradients = None
activations = None

def backward_hook(module, grad_input, grad_output):

    global gradients

    gradients = grad_output[0]

def forward_hook(module, input, output):

    global activations

    activations = output

# ============================================================
# TARGET LAYER
# ============================================================

target_layer = model.densenet.features.denseblock4

target_layer.register_forward_hook(
    forward_hook
)

target_layer.register_full_backward_hook(
    backward_hook
)

# ============================================================
# FORWARD PASS
# ============================================================

output = model(input_tensor)

probabilities = torch.softmax(
    output,
    dim=1
)

pred_class = output.argmax(dim=1)

predicted_class = class_names[
    pred_class.item()
]

print("Prediction:", predicted_class)

# ============================================================
# BACKWARD PASS
# ============================================================

model.zero_grad()

loss = output[0, pred_class]

loss.backward()

# ============================================================
# GRAD-CAM
# ============================================================

pooled_gradients = torch.mean(
    gradients,
    dim=[0,2,3]
)

activation = activations.detach().squeeze()

for i in range(len(pooled_gradients)):

    activation[i,:,:] *= pooled_gradients[i]

heatmap = torch.mean(
    activation,
    dim=0
).cpu().numpy()

heatmap = np.maximum(
    heatmap,
    0
)

heatmap /= heatmap.max()

# ============================================================
# RESIZE HEATMAP
# ============================================================

heatmap = cv2.resize(
    heatmap,
    (224,224)
)

# ============================================================
# COLOR HEATMAP
# ============================================================

heatmap_color = cv2.applyColorMap(

    np.uint8(
        255 * heatmap
    ),

    cv2.COLORMAP_JET
)

heatmap_color = cv2.cvtColor(
    heatmap_color,
    cv2.COLOR_BGR2RGB
)

# ============================================================
# OVERLAY
# ============================================================

overlay = cv2.addWeighted(

    original_np,

    0.6,

    heatmap_color,

    0.4,

    0
)

# ============================================================
# CONFIDENCE
# ============================================================

confidence = (
    probabilities
    .detach()
    .cpu()
    .numpy()[0]
)

# ============================================================
# VISUALIZATION
# ============================================================

fig = plt.figure(
    figsize=(20,8)
)

fig.suptitle(
    f"Grad-CAM Visual Explanation\nPrediction: {predicted_class}",
    fontsize=18
)

# ============================================================
# ORIGINAL IMAGE
# ============================================================

ax1 = plt.subplot(1,4,1)

ax1.imshow(original_np)

ax1.set_title("Original X-ray")

ax1.axis("off")

# ============================================================
# ATTENTION HEATMAP
# ============================================================

ax2 = plt.subplot(1,4,2)

ax2.imshow(
    heatmap,
    cmap='jet'
)

ax2.set_title("Attention Heatmap")

ax2.axis("off")

# ============================================================
# FINAL OVERLAY
# ============================================================

ax3 = plt.subplot(1,4,3)

ax3.imshow(overlay)

ax3.set_title("Model Focus Region")

ax3.axis("off")

# ============================================================
# CONFIDENCE BAR
# ============================================================

ax4 = plt.subplot(1,4,4)

ax4.bar(
    class_names,
    confidence
)

ax4.set_ylim(0,1)

ax4.set_ylabel("Probability")

ax4.set_title("Prediction Confidence")

# ============================================================
# SAVE RESULT
# ============================================================

plt.tight_layout()

save_path = os.path.join(
    output_dir,
    "gradcam_visualization.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("Saved at:", save_path)